<a href="https://colab.research.google.com/github/OmarZED/Auth-System-with-Express/blob/main/ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
# 1. Clean up
!rm -rf train valid test data.yaml
# 2. Download from Roboflow
!curl -L "https://app.roboflow.com/ds/GsarTiJzjv?key=fLKENiicPO" > roboflow.zip; unzip -q roboflow.zip; rm roboflow.zip
# 3. Setup Folders
if os.path.exists('val') and not os.path.exists('valid'):
    os.rename('val', 'valid')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   927  100   927    0     0   3058      0 --:--:-- --:--:-- --:--:--  3069
100 4633M  100 4633M    0     0  80.9M      0  0:00:57  0:00:57 --:--:-- 77.5M


In [9]:
import os
import random
import shutil

# 1. Define the paths based on your folder structure
train_img = '/content/train/images'
train_lbl = '/content/train/labels'
valid_img = '/content/valid/images'
valid_lbl = '/content/valid/labels'

# 2. Create the new folders
os.makedirs(valid_img, exist_ok=True)
os.makedirs(valid_lbl, exist_ok=True)

# 3. Get list of all images
images = [f for f in os.listdir(train_img) if f.endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(images)

# 4. Move 2,000 images and their matching labels
images_to_move = images[:2000]

print(f"📦 Moving {len(images_to_move)} image/label pairs to validation...")

for img_name in images_to_move:
    # Move Image
    shutil.move(os.path.join(train_img, img_name), os.path.join(valid_img, img_name))

    # Move matching Label (.txt)
    lbl_name = os.path.splitext(img_name)[0] + '.txt'
    if os.path.exists(os.path.join(train_lbl, lbl_name)):
        shutil.move(os.path.join(train_lbl, lbl_name), os.path.join(valid_lbl, lbl_name))

print("✅ Validation split complete!")

📦 Moving 2000 image/label pairs to validation...
✅ Validation split complete!


In [11]:
import yaml

data_config = {
    # Path to your training images
    'train': '/content/train/images',

    # Path to your validation images (the folder we just created)
    'val': '/content/valid/images',

    'nc': 14,
    'names': [
        'Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest',
        'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'glove',
        'goggles', 'no-glove', 'no-goggles', 'fall-detected'
    ]
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("✅ data.yaml is now pointing to your two separate folders correctly.")

✅ data.yaml is now pointing to your two separate folders correctly.


In [7]:
!pip install ultralytics roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.6 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

model.train(
    data='/content/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    amp=False,
    project='Suez-Safety-Final',
    name='MVP_Stable_Run'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, i